# Week 5 Advanced — LQR, Kalman, LQG, and Constrained MPC on One Plant

Everything now acts on the **same** three-state propulsion-style plant instead of separate toy systems.

State vector: $$x=[\omega,\ T_{act},\ z]^T$$
where $z$ is a slow internal state representing something like thermal/powertrain lag.

We will:
1. design LQR using full state truth,
2. hide states and estimate them with a Kalman filter,
3. close LQG using estimated state,
4. introduce model mismatch and sensor noise,
5. compare against constrained MPC when actuator/state limits bind.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_continuous_are, expm, solve_discrete_are

A=np.array([[-0.35, 8.0, 0.0],
            [ 0.0,-20.0, 3.0],
            [ 0.0, 0.0,-0.5]])
B=np.array([[0.0],[20.0],[0.8]])
C=np.array([[1.0,0.0,0.0]])
Q=np.diag([20.0,1.0,3.0]); R=np.array([[0.8]])
P=solve_continuous_are(A,B,Q,R); K=np.linalg.solve(R,B.T@P)
print('Open-loop poles:',np.linalg.eigvals(A))
print('LQR gain:',K)
print('Closed-loop poles:',np.linalg.eigvals(A-B@K))


## 1. Full-state LQR benchmark

This is the optimistic case: the controller knows every state exactly and has no actuator limits.

In [ ]:
dt=0.002; t=np.arange(0,8,dt); x=np.array([8.0,0.0,2.0]); H=[]
for ti in t:
    d=np.array([2.0 if ti>3 else 0.0,0,0])
    u=float(-K@x)
    x += (A@x+B[:,0]*u+d)*dt
    H.append([*x,u])
H=np.array(H)
plt.figure(figsize=(9,4)); plt.plot(t,H[:,:3]); plt.legend(['omega','Tact','z']); plt.grid(True); plt.show()


## 2. Discretize and build a Kalman estimator

Only $\omega$ is measured. The estimator must reconstruct $T_{act}$ and $z$ from the model and the measurement history.

In [ ]:
Ts=0.01
M=np.block([[A,B],[np.zeros((1,4))]])
Md=expm(M*Ts); Ad=Md[:3,:3]; Bd=Md[:3,3:4]
Qn=np.diag([2e-3,8e-3,2e-3]); Rn=np.array([[0.05]])
# steady-state discrete Kalman gain via dual DARE
Pk=solve_discrete_are(Ad.T,C.T,Qn,Rn)
L=Pk@C.T@np.linalg.inv(C@Pk@C.T+Rn)
print('Kalman gain:',L.ravel())


## 3. LQG with noise and model mismatch

The real plant below is deliberately 15% different from the estimator model. This is where 'optimal' stops meaning 'perfect'.

In [ ]:
rng=np.random.default_rng(7)
Areal=A.copy(); Areal[1,1]*=0.85; Areal[2,2]*=1.15
Mreal=np.block([[Areal,B],[np.zeros((1,4))]])
Mrd=expm(Mreal*Ts); Ard=Mrd[:3,:3]; Brd=Mrd[:3,3:4]
x=np.array([8.,0.,2.]); xhat=np.zeros(3); H=[]
for k in range(800):
    u=float(np.clip(-K@xhat,-2.5,2.5))
    x=Ard@x+Brd[:,0]*u+rng.multivariate_normal(np.zeros(3),Qn)
    y=float(C@x+rng.normal(0,np.sqrt(Rn[0,0])))
    xpred=Ad@xhat+Bd[:,0]*u
    xhat=xpred+(L@(np.array([y])-C@xpred)).ravel()
    H.append([*x,*xhat,u,y])
H=np.array(H); tt=np.arange(len(H))*Ts
plt.figure(figsize=(9,4)); plt.plot(tt,H[:,0],label='true omega'); plt.plot(tt,H[:,3],label='estimated omega'); plt.grid(True); plt.legend(); plt.show()
plt.figure(figsize=(9,4)); plt.plot(tt,H[:,2],label='true hidden z'); plt.plot(tt,H[:,5],label='estimated z'); plt.grid(True); plt.legend(); plt.show()


## 4. Why MPC becomes interesting

LQR says $u=-Kx$ and only learns about limits after you clip the command. MPC includes limits **inside the optimization**.

For this exercise, impose
$$|u|\le2.5,\qquad |T_{act}|\le2.0,\qquad |z|\le3.0.$$

Instead of hiding the optimization in a package, use a short-horizon candidate search to see the mechanism directly.

In [ ]:
u_grid=np.linspace(-2.5,2.5,11)
def mpc_first_action(x0,N=3):
    best=(np.inf,0.0)
    from itertools import product
    for seq in product(u_grid,repeat=N):
        x=x0.copy(); cost=0.; feasible=True
        for u in seq:
            x=Ad@x+Bd[:,0]*u
            if abs(x[1])>2.0 or abs(x[2])>3.0:
                feasible=False; break
            cost += x@Q@x + float(R[0,0]*u*u)
        if feasible and cost<best[0]: best=(cost,seq[0])
    return best[1]

x=np.array([8.,0.,2.]); Xm=[]; Um=[]
for k in range(150):
    u=mpc_first_action(x)
    x=Ad@x+Bd[:,0]*u
    Xm.append(x.copy()); Um.append(u)
Xm=np.array(Xm)
plt.figure(figsize=(9,4)); plt.plot(Xm); plt.legend(['omega','Tact','z']); plt.grid(True); plt.show()
print('max |u|=',np.max(np.abs(Um)),'max |Tact|=',np.max(np.abs(Xm[:,1])))


## Engineering tasks

- Increase measurement noise by 10x. Which hidden state estimate degrades first? Why?
- Increase process noise while leaving $R_n$ fixed. Explain how the Kalman gain should move.
- Compare full-state LQR, clipped LQR, LQG, and MPC using settling time, RMS control effort, and constraint violations.
- Tighten the $T_{act}$ limit until clipped LQR performs badly. Does MPC anticipate the limit better?
- Change the plant by 30% and identify when estimator/controller model mismatch becomes unacceptable.

**Deliverable:** a short trade study answering when LQR/LQG is sufficient and when explicit MPC constraint handling earns its complexity.